### WEB Crawling

In [97]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import PorterStemmer
import string

In [2]:
def mycrawler(seed, maxcount):
    Q = [seed] #this is the queue which initially contains the given seed URL
    count = 0
    while(Q!=[] and count < maxcount):
        count +=1
        url = Q[0]
        Q = Q[1:]
        #print("fetching " + url)

        code = requests.get(url)
        plain = code.text
        s = BeautifulSoup(plain, "html.parser")
        for link in s.findAll('a'):
            new_url = link.get('href')
            if(new_url != None and new_url != '/'):
                new_url = new_url.strip()
                #print("new_url is : ", new_url)

                #normalise if needed
                if( new_url[0:7] != 'http://' and new_url[0:8]!='https://' ) :
                    if(url[len(url) -1] == '/'):
                        if(new_url[0] == '/'):
                            url_split = new_url.split('/')
                            #print(url_split)
                            for k in range(len(url_split)):
                                if(url_split[k] == 'publications'):
                                    #print(seed+'/'+url_split[k])
                                    new_url=seed+'/'+url_split[k]+'/'
                                    return new_url
                            #print('URL : ',url)
                            #print('NEW URL : ',new_url[0:len(new_url)])
                            new_url = url + new_url[1:len(new_url)]
                            #print('new new url is : ',new_url)
                    else:
                        new_url = url + '/' + new_url
                                #NOTE : further normalization code required here
                #print("new_url is : ", new_url)  #uncomment the print statement to see the urls
            Q.append(new_url)
            
#mycrawler('https://scholar.google.com',10) #use any number, instead of 10, to control the number of pages crawled
#seed = https://pureportal.coventry.ac.uk/en/organisations/school-of-mechanical-aerospace-and-automotive-engineering/
research_publications=mycrawler('https://pureportal.coventry.ac.uk/en/organisations/school-of-mechanical-aerospace-and-\
                                automotive-engineering/',10)
print(research_publications)

https://pureportal.coventry.ac.uk/en/organisations/school-of-mechanical-aerospace-and-automotive-engineering//publications/


In [3]:
def make_soup(url):
    html = requests.get(url)
    plain = html.text
    soup = BeautifulSoup(plain, 'html.parser')
    return soup

def crawler(seed):
    
    Q = [seed]
    publication_title = []
    authors = []
    pub_year = []
    pub_url = []
    EEC_author_proflink = []
    
    for page in range (0,34):
        if page == 0: 
            page_no =  1
        else:
            page_url = seed +'?page='+str(page)
            Q.append(page_url)
            
    while(Q!=[]):
        next_url = Q[0]
        Q = Q[1:]
        
        print('CRAWLER IS CURRENTLY ON THIS LINK', next_url)
        page_soup = make_soup(next_url)
        publications = page_soup.find_all("div",{'class':'result-container'})
        
        for pub in publications:
            
            #find publication year
            date = pub.find('span', {'class':'date'}).text
            year = date[-4:]
            pub_year.append(year)
            print('--------------------------------------------------------------------------------------------------')
            print("Publication Year: ", year)
            
            #find publication URL link
            publication_link = pub.find('a', {'class':'link'}).get('href')
            pub_url.append(publication_link)
            print("Publication URL link: ", publication_link)
            pub_soup = make_soup(publication_link)
            
            #find publication title 
            title_pub = pub_soup.find('div', {'class':'rendering'}).text
            publication_title.append(title_pub)
            print("Publication Title: ", title_pub)
            
            #find author/s
            authors_pub = pub_soup.find('div', {'class':'rendering_researchoutput'}).text
            authors.append(authors_pub)
            print("Publication Authors: ", authors_pub)
            
            #find author profile link
            profile_link = pub_soup.find_all('a', {'class':'link person'})
            person_profile = []
            for profile in profile_link:
                person = profile.get('href')
                person_profile.append(person)
                print("EEC Author's profile link: ", person)
                person_profile = list(dict.fromkeys(person_profile))
            EEC_author_proflink.append(person_profile)
            print('-------------------------------------------------------------------------------------------------')
            
    Doc_ID = list(range(1,len(publication_title)+1))

    Results = {'Doc_ID'                   : Doc_ID,
               'Title'                    : publication_title,
               'Authors'                  : authors,
               'Year'                     : pub_year,
               'Publication url'          : pub_url,
               'EEC Author Profile links' : EEC_author_proflink}

    df_results = pd.DataFrame(Results)
    #df_results.to_csv('C:/Users/Abrar Ahmad/Desktop/file_name.csv',index=False,encoding='utf-8')

    return(publication_title,authors,pub_year,pub_url,EEC_author_proflink,df_results)


In [4]:
crawler(research_publications)

CRAWLER IS CURRENTLY ON THIS LINK https://pureportal.coventry.ac.uk/en/organisations/school-of-mechanical-aerospace-and-automotive-engineering//publications/
--------------------------------------------------------------------------------------------------
Publication Year:  2021
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/a-bibliometric-review-on-the-implications-of-renewable-offshore-m
Publication Title:  A bibliometric review on the implications of renewable offshore marine energy development on marine species
Publication Authors:  Siddharth Kulkarni, David Edwards
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/siddharth-kulkarni
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  2021
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/a-comparat

Publication Title:  A Multi Agent-Based Optimisation Model for the Distribution Planning and Control of Energy-Based Intermittent Renewable Sources
Publication Authors:  Nooria Mohammed, Ammar Al Bazi
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/ammar-al-bazi
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  2021
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/an-adaptive-backpropagation-algorithm-for-long-term-electricity-l
Publication Title:  An Adaptive Backpropagation Algorithm for Long Term Electricity Load Forecasting
Publication Authors:  Nooria Mohammed, Ammar Al Bazi
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/ammar-al-bazi
-------------------------------------------------------------------------------------------------
-------------

Publication Title:  A novel energy harvesting approach for hybrid electromagnetic-based suspension system of off-road vehicles considering terrain deformability
Publication Authors:  Hamid Taghavifar
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/hamid-taghavifar
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  2021
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/a-novel-optimal-path-planning-and-following-algorithm-for-wheeled
Publication Title:  A novel optimal path-planning and following algorithm for wheeled robots on deformable terrains
Publication Authors:  Hamid Taghavifar, Subhash Rakheja, Giulio Reina
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/hamid-taghavifar
--------------------------------------------------------------------------

Publication Title:  Cold-start NOx emissions: Diesel and waste lubricating oil as a fuel additive
Publication Authors:  Ali Zare, Timothy A. Bodisco, Mohammad Jafari, Puneet Verma, Liping Yang, Meisam Babaie, M. M. Rahman, Andrew Banks, Zoran D. Ristovski, Richard J. Brown, Svetlana Stevanovic
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/mostafiz-rahman
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  2021
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/combustion-of-fuel-surrogates-an-application-to-gas-turbine-engin
Publication Title:  Combustion of Fuel Surrogates: An application to gas turbine engines
Publication Authors:  Mansour Al Qubeissi, Nawar Al-Esawi, Hakan Serhad Soyhan
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/m-al-qubeissi
-

Publication Title:  EKF-Neural Network Observer Based Type-2 Fuzzy Control of Autonomous Vehicles
Publication Authors:  Hamid Taghavifar, Chuan Hu, Yechen Qin, Chongfeng Wei
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/hamid-taghavifar
-------------------------------------------------------------------------------------------------
CRAWLER IS CURRENTLY ON THIS LINK https://pureportal.coventry.ac.uk/en/organisations/school-of-mechanical-aerospace-and-automotive-engineering//publications/?page=1
--------------------------------------------------------------------------------------------------
Publication Year:  2021
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/energy-efficient-double-pass-photovoltaicthermal-air-systems-usin
Publication Title:  Energy efficient double-pass photovoltaic/thermal air systems using a computational fluid dynamics multi-objective optimisation framework
Publication Authors:  Moustafa Al-Damook, Zinedine Kh

...

Publication Title:  Design and manufacture of a head and neck phantom for the assessment of IMRT delivery with gel dosimetry
Publication Authors:  J. Meyer, P. Harding, J. Mills, D. Bonnett, T. Wolff, O.C.L. Haas, Keith Burnham
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/olivier-haas
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  2001
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/design-of-a-bilinear-compensator-for-industrial-furnaces-2
Publication Title:  Design of a bilinear compensator for industrial furnaces
Publication Authors:  S. Martineau, Keith J. Burnham, J. Minihan, Elena Gaura, Olivier C.L. Haas
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/elena-gaura
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/o

Publication Title:  Intelligent systems for optimisation and control
Publication Authors:  Keith J. Burnham, Olivier C.L. Haas, D.J.G. James
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/olivier-haas
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  2000
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/some-limitations-in-the-practical-delivery-of-intensity-modulated-2
Publication Title:  Some limitations in the practical delivery of intensity modulated radiation therapy
Publication Authors:  J. Meyer, J.A. Mills, Olivier C.L. Haas, E. Parvin, Keith J. Burnham
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/olivier-haas
-------------------------------------------------------------------------------------------------
--------------------------------

Publication Title:  On Supervisory Control System Design with Application to Power Generation Plants. PhD Thesis, Industrial Control Centre, Department of Electronic and Electrical Engineering, University of Strathclyde, Glasgow,
Publication Authors:  Andrew Pike
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/andrew-pike
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  1998
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/optimization-of-beam-orientation-in-radiotherapy-using-planar-geo-2
Publication Title:  Optimization of beam orientation in radiotherapy using planar geometry
Publication Authors:  Olivier Haas, Keith Burnham, J.A. Mills
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/olivier-haas
--------------------------------------------------

Publication Title:  Use of genetic algorithms in radiotherapy treatment planning
Publication Authors:  Olivier C.L. Haas, Keith J. Burnham, M.H. Fisher, John A. Mills
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/olivier-haas
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------
Publication Year:  1995
Publication URL link:  https://pureportal.coventry.ac.uk/en/publications/constrained-control-for-power-generation-plant
Publication Title:  Constrained Control for Power Generation plant
Publication Authors:  Reza Katebi, Mike Johnson, Andrew Pike, Andrzej Ordys
EEC Author's profile link:  https://pureportal.coventry.ac.uk/en/persons/andrew-pike
-------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------

(['A bibliometric review on the implications of renewable offshore marine energy development on marine species',
  'A comparative review of hand-eye calibration techniques for vision guided robots',
  'Acoustic radiation modes and modal criterion dependency of a planetary gearbox for fault detection under the presence of tooth flank fracture',
  'A critique of the THUMS lower limb model for pedestrian impact applications',
  'Activation of nano kaolin clay for bio-glycerol conversion to a valuable fuel additive',
  'Adaptive robust control-based energy management of hybrid PV-Battery systems with improved transient performance',
  'A Decision Process for the Applications of Artificial Intelligence in Sustainable Operations and Supply Chain Management',
  'A Generic Brain Trauma Computer Framework to Assess Brain Injury Severity and Bridging Vein Rupture in Traumatic Falls',
  'A Jump-Markov Regularized Particle Filter for the estimation of ambiguous sensor faults',
  'A Machine Learnin

In [141]:
df_results = pd.read_csv ('file_name.csv')
#print (df_results)
df_results.head

<bound method NDFrame.head of       Doc_ID                                              Title  \
0          1  A comparative review of hand-eye calibration t...   
1          2  Acoustic radiation modes and modal criterion d...   
2          3  A critique of the THUMS lower limb model for p...   
3          4  Activation of nano kaolin clay for bio-glycero...   
4          5  Adaptive robust control-based energy managemen...   
...      ...                                                ...   
1677    1678  A Coordination Strategy for Integrated Operati...   
1678    1679                 Microexcavator design optimisation   
1679    1680               Computer modelling of microexcavator   
1680    1681  Rapid data acquisition using a data logger int...   
1681    1682            Structural analysis of a microexcavator   

                                                Authors  Year  \
0     Ikenna Enebuse, Mathias Foo, Babul Salam Ksm K...  2021   
1     Imthiyas Manarikkal, Faris El

### INDEXER

In [7]:
#df_results.columns
filtered_titles = df_results['Title']
#print(filtered_titles)
cleaned_doc = []
for doc in filtered_titles:
    text_doc = re.sub(r'[^\x00-\x7F]+',' ', doc)
    text_doc = text_doc.lower()
    text_doc = re.sub(r'@\w+','', text_doc)
    text_doc = re.sub(r'[^\w\s]','', text_doc)
    text_doc = re.sub(r'[0-9]','', text_doc)
    text_doc = re.sub(r'\s{2,}','', text_doc)
    
    remove_stopwords = re.compile(r'\b('+ r'|'.join(stopwords.words('english')) +r')\b\s*')
    text = remove_stopwords.sub('',text_doc)
    cleaned_doc.append(text)
    
vectorizer = TfidfVectorizer()
vectorized_doc = vectorizer.fit_transform(cleaned_doc)
vectorized_doc = vectorized_doc.T.toarray()
inv_df = pd.DataFrame(vectorized_doc, index=vectorizer.get_feature_names())
print('inverted index created')



print(cleaned_doc)
filtered = pd.DataFrame({'filtered':cleaned_doc})
print(filtered)
df_results['filtered'] = filtered['filtered']
print(df_results)

inverted index created
['comparative review handeye calibration techniques vision guided robots', 'acoustic radiation modes modal criterion dependency planetary gearbox fault detection presence tooth flank fracture', 'critique thums lower limb model pedestrian impact applications', 'activation nano kaolin clay bioglycerol conversion valuable fuel additive', 'adaptive robust controlbased energy management hybrid pvbattery systems improved transient performance', 'decision process applications artificial intelligence sustainable operations supply chain management', 'generic brain trauma computer framework assess brain injury severity bridging vein rupture traumatic falls', 'jumpmarkov regularized particle filter estimation ambiguous sensor faults', 'multi agentbased optimisation model distribution planning control energybased intermittent renewable sources', 'adaptive backpropagation algorithm long term electricity load forecasting', 'ahp fuzzy ahp multifactor decision making approach te

In [93]:
inv_df.head()

,0,1,2,3,4,5,6,7,8,9,...,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681
aal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aaresta,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
aarestrendezvous,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ab,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ablation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [111]:
def indexer(filtered_titles, df_results):
    
    index_dic = {}
    
    for each_word in cleaned_doc:
        words = word_tokenize(each_word)
        
        for word in words:
            if word not in index_dic.keys():
                index_dic[word] = []
                #print(index_dic)
                
#find the publications in which each of the key word exists
    for i in range(len(df_results)):
        pub_name_keywords = df_results.loc[i, "filtered"]
        pub_ID = df_results.loc[i,"Doc_ID"]
        x = pub_name_keywords.split()
        
        for w in x:
            if w in index_dic:
                if pub_ID not in index_dic[w]:
                    index_dic[w].append(pub_ID)
                    
    dictionary_items = index_dic.items()
    
    sorted_index_list = sorted(dictionary_items)
    sorted_index = {}
    
    for (key,value) in sorted_index_list:
        sorted_index.update({key : value})
        
    return(sorted_index)

sorted_index = indexer(filtered_titles, df_results)
print(sorted_index)
                    

{'aal': [402], 'aaresta': [1177], 'aarestrendezvous': [1178], 'ab': [670, 1091], 'ablation': [147, 150, 329, 1226, 1448], 'abnormal': [202, 301], 'absence': [638], 'absorbing': [1541], 'absorption': [777, 1669], 'absorptiondesorption': [1218], 'abstraction': [330], 'academic': [839], 'acausal': [1171], 'accelerated': [577, 578], 'acceleration': [597, 598], 'access': [13, 436, 915], 'accident': [661, 1367], 'accidents': [1461], 'accommodation': [1623], 'accumulation': [454], 'accuracy': [108, 285, 1210, 1333], 'accurate': [469, 471, 690], 'acenter': [719, 1148], 'achieve': [569, 940], 'achieved': [1458], 'achievements': [836], 'achieving': [1231, 1661], 'acid': [193, 308, 1293], 'acidbased': [463], 'acidified': [135], 'acids': [167, 213], 'acos': [1512], 'acoustic': [2, 103, 430, 450, 473, 505, 671, 757, 884, 916, 1034, 1498], 'acoustics': [1044], 'acoustofluidic': [226], 'acquisition': [1681], 'across': [951, 1080, 1275], 'action': [719, 1148], 'activated': [203, 956, 1083, 1091, 1229,

### QUERY SEARCH

In [137]:
def query_processor(inserted_query,sorted_index,df_results):
    
    query_prep = re.sub(r'[^\x00-\x7F]+',' ', inserted_query) # Remove Unicode
    query_prep = re.sub(r'@\w+','', query_prep)  # Remove Mentions
    query_prep = query_prep.lower()  # Lowercase whole document
    query_prep = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', query_prep) # Removing punctuations
    query_clean = re.sub(r'\s{2,}','', text_doc) # Removing double space
    
    # Removing stopwords, Tokenizing and stemming
    stopword = stopwords.words('english')
    stemmer = PorterStemmer()
    filtered_query = []
    for q in query_clean:
        tokens = word_tokenize(q)
        tmp = ""
        for w in tokens:
            if w not in stopword:
                tmp += stemmer.stem(w) + " "
        filtered_query.append(tmp)
        #print(filtered_query)
            
    # Search the index to find each publication that has atleast on of the given words in the query          
    for item in filtered_query:
        all_words = item.split()
        relevant_docs = []
        for word in all_words:
            if word in sorted_index:
                post_list = sorted_index[word]
                for post in post_list:
                    relevant_docs.append(post)
                    
    relevant_docs = list(set(relevant_docs))
    #print(relevant_docs)
    
    
    # Obtain index for the relevant docs, subtract 1 from the DocID
    relevant_docs_index = []
    for i in relevant_docs:
        relevant_docs_index.append(i-1)
        #print(relevant_docs_index)
        
    # Creating a dataframe of just relevant docs
        df_relevant_docs = df_results.iloc[relevant_docs_index]
        #print(df_relevant_docs.shape)
    
    
    # Make a list of relevant pre-processed titles
    relevant_filtered_titles = df_relevant_docs['filtered'].to_list()
    
    # Vectorizing relevant filtered titles and filtered query
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(relevant_filtered_titles)
    y = vectorizer.transform(filtered_query)
    
    # Calculate Cosine similarity
    cos_sim = cosine_similarity(X,y).reshape((-1,))
    
    # Ranking the most relevant result (Number 1) to to the least relevant result
    ranking = list(range(len(relevant_docs)))
    
    # Collecting the information for the searched query in order of relevance
    publication_title = []
    authors = []
    pub_year = []
    pub_url = []
    EEC_author_proflink = []
    for i in cos_sim.argsort()[-(len(relevant_docs)):][::-1]:
        title_result = df_relevant_docs.iloc[:i,1]
        publication_title.append(title_result)
        authors_result =  df_relevant_docs.iloc[:i,2]
        authors.append(authors_result)
        pub_year_result =  df_relevant_docs.iloc[:i,3]
        pub_year.append(pub_year_result)
        pub_url_result = df_relevant_docs.iloc[:i,4]
        pub_url.append(pub_url_result)
        EEC_proflink = df_relevant_docs.iloc[:i,5]
        EEC_author_proflink.append(EEC_proflink)
        
    
    for n in ranking:
        print('------------------------------The result is:',n+1,'-----------------------------------------------')
        print('Publication Title'        , publication_title[n])
        print('Publication Authors'      , authors[n])
        print('Publication Year'         , pub_year[n])
        print('Publication url'          , pub_url[n])
        print('EEC Author Profile links:')
        for j in EEC_author_proflink[n]:
            print(j)
        print('--------------------------------------------------------------------------------------------------')
        
    return("Total results", len(relevant_docs))
        

In [139]:
inserted_query = input("Please enter your query: ")
print(query_processor(inserted_query,sorted_index,df_results))

Please enter your query: Direct current interruption
------------------------------The result is: 1 -----------------------------------------------
Publication Title 1184    Direct current interruption with R L C circuit...
Name: Title, dtype: object
Publication Authors 1184    Leon Shpanin, N. Y. A. Shammas, S. B. Tennakoo...
Name: Authors, dtype: object
Publication Year 1184    2013
Name: Year, dtype: int64
Publication url 1184    https://pureportal.coventry.ac.uk/en/publicati...
Name: Publication url, dtype: object
EEC Author Profile links:
[]
--------------------------------------------------------------------------------------------------
('Total results', 1)
